# Deep Challenger Benchmark

This notebook is the scaffold for the main deep benchmark across the three datasets.

Benchmark families:

- `SARIMAX`
- `HURDLE`
- `TSB`
- `Croston-SBA`
- existing repo challengers handled elsewhere: `LSTM`, `TCN`
- new missing challengers added in code: `GRU`, `DeepAR`

## Notes

- In this notebook, prioritize only the missing deep models for comparison.
- Existing repo work already covers `LSTM` and `TCN` in older notebooks.
- Evaluate by dataset and by regime.
- Preserve the regime-aware framing of the project.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / 'src').exists() and (cur / 'reports').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Repo root:', REPO_ROOT)

In [ ]:
from src.data_loaders.load_m5 import load_m5_single_series
from src.data_loaders.load_favorita import load_favorita_series
from src.data_loaders.load_amazon import load_amazon_series
from src.models.deep_challenger_models import GRUForecastModel, DeepARForecastModel, train_test_split_series, mae, rmse, smape
from src.metrics.behavioral_metrics import behavioral_metrics

In [ ]:
benchmark_cases = {
    'm5': load_m5_single_series(base_path='data/raw/m5', random_pick=True, seed=42),
    'favorita': load_favorita_series(base_path='data/raw/favorita', store_nbr=1, family='CLEANING'),
    'amazon': load_amazon_series(
        base_path='data/raw/amazon_2023/review_categories',
        filename='Health_and_Household.jsonl.gz',
        top_rank=1,
        max_rows=100000,
    ),
}

for name, df in benchmark_cases.items():
    print(name, df.shape)

In [ ]:
deep_model_registry = {
    'GRU': {'status': 'ready', 'notes': 'implemented in src.models.deep_challenger_models'},
    'DeepAR': {'status': 'ready', 'notes': 'implemented in src.models.deep_challenger_models'},
}

pd.DataFrame(deep_model_registry).T

In [ ]:
def evaluate_deep_case(df, model_name, model):
    df = df.sort_values('date').reset_index(drop=True)
    y = pd.to_numeric(df['sales'], errors='coerce').fillna(0.0).to_numpy(dtype=float)
    y_train, y_test = train_test_split_series(y, test_days=365)
    model.fit(y_train)
    y_pred, _, _ = model.forecast(len(y_test), y_train)
    row = {
        'model': model_name,
        'train_days': len(y_train),
        'test_days': len(y_test),
        'mae': mae(y_test, y_pred),
        'rmse': rmse(y_test, y_pred),
        'smape': smape(y_test, y_pred),
    }
    row.update(behavioral_metrics(y_test, y_pred))
    return row

In [ ]:
example_results = []
for dataset_name, df in benchmark_cases.items():
    for model_name, model in {
        'GRU': GRUForecastModel(context_length=28, hidden_size=32, epochs=10),
        'DeepAR': DeepARForecastModel(context_length=28, hidden_size=32, epochs=10),
    }.items():
        row = evaluate_deep_case(df, model_name, model)
        row['dataset'] = dataset_name
        example_results.append(row)

pd.DataFrame(example_results)

In [ ]:
benchmark_result_columns = [
    'dataset',
    'series_id',
    'regime',
    'model',
    'MAE',
    'RMSE',
    'sMAPE',
]

results_df = pd.DataFrame(columns=benchmark_result_columns)
results_df